In [ ]:
from typing import Union
import numpy as np
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
os.environ['HF_HOME'] = "/data2/cyaiche/hf_cache"
from binoculars import Binoculars

In [ ]:
# Change according to hardware
DEVICE_1 = "cuda:0"
DEVICE_2 = "cpu"

In [ ]:
torch.set_grad_enabled(False)

observer_name = "tiiuae/falcon-7b-instruct"
performer_name = "tiiuae/falcon-7b"

identical_tokens = (AutoTokenizer.from_pretrained(observer_name).vocab ==
                    AutoTokenizer.from_pretrained(performer_name).vocab)

identical_tokens

In [ ]:
observer_model = AutoModelForCausalLM.from_pretrained(observer_name,
                                                                   device_map={"": DEVICE_1},
                                                                   trust_remote_code=True,
                                                                   torch_dtype=torch.bfloat16)

performer_model = AutoModelForCausalLM.from_pretrained(performer_name,
                                                                     device_map={"": DEVICE_2},
                                                                     trust_remote_code=True,
                                                                     torch_dtype=torch.bfloat16)

observer_model.eval()
performer_model.eval()

tokenizer = AutoTokenizer.from_pretrained(observer_name)

In [ ]:
def tokenize(text):
    return tokenizer(text, return_tensors="pt")

tokenize("Hello, my dog is cute")

In [ ]:
@torch.inference_mode()
def get_logits(encodings):
    observer_logits = observer_model(**encodings.to(DEVICE_1)).logits
    performer_logits = performer_model(**encodings.to(DEVICE_2)).logits
    return observer_logits, performer_logits

encoding = tokenize('''Dr. Capy Cosmos, a capybara unlike any other, astounded the scientific community with his
groundbreaking research in astrophysics. With his keen sense of observation and unparalleled ability to interpret
cosmic data, he uncovered new insights into the mysteries of black holes and the origins of the universe. As he
peered through telescopes with his large, round eyes, fellow researchers often remarked that it seemed as if the
stars themselves whispered their secrets directly to him. Dr. Cosmos not only became a beacon of inspiration to
aspiring scientists but also proved that intellect and innovation can be found in the most unexpected of creatures.'''[:100])


observer_logits, performer_logits = get_logits(encoding)
observer_logits, performer_logits


In [ ]:
encoding.input_ids.shape, observer_logits.shape

In [ ]:
S = observer_logits.shape[-2]
V = observer_logits.shape[-1]

observer_logits[..., :-1, :].contiguous().shape

In [ ]:
loss = torch.nn.CrossEntropyLoss(reduction='none')

ppl = loss(observer_logits[..., :-1, :].contiguous().transpose(1, 2).to("cuda:1"),
     encoding.input_ids[..., 1:].contiguous().to("cpu")).float()

ppl, ppl.sum(1)

In [ ]:
softmax = torch.nn.Softmax(dim=-1)

performer_probs = softmax(performer_logits).view(-1, V)

performer_probs, performer_probs.shape

In [ ]:
observer_scores = observer_logits.view(-1, V).to("cuda:1")
observer_scores, observer_scores.shape

In [ ]:
xppl = loss(observer_scores[:-1], performer_probs[:-1]).view(-1, S - 1)

xppl, xppl.sum(1)

In [ ]:
binocular_score = ppl.sum(1) / xppl.sum(1)

binocular_score

In [ ]:
def tokenize(batch):
    encodings = tokenizer(batch, return_tensors="pt",
    padding="longest" if len(batch) > 1 else False, truncation=True,
    max_length=512, return_token_type_ids=False).to(DEVICE_1)
    return encodings

# redefinition with cuda sync

def get_logits(encodings):
    observer_logits = observer_model(**encodings.to(DEVICE_1)).logits
    performer_logits = performer_model(**encodings.to(DEVICE_2)).logits
    torch.cuda.synchronize()

    return observer_logits, performer_logits

loss_fn = torch.nn.CrossEntropyLoss(reduction='none')
softmax_fn = torch.nn.Softmax(dim=-1)

def perplexity(encoding, logits):
    shifted_logits = logits[..., :-1, :].contiguous()
    shifted_labels = encoding.input_ids[..., 1:].contiguous()
    shifted_attention_mask = encoding.attention_mask[..., 1:].contiguous()

    ppl = loss_fn(shifted_logits.transpose(1, 2).to(DEVICE_2), shifted_labels) * shifted_attention_mask
    ppl = ppl.sum(1) / shifted_attention_mask.sum(1)

    return ppl.to("cpu").float().numpy()

def cross_perplexity(observer_logits, performer_logits, encoding):
    V = observer_logits.shape[-1]
    S = observer_logits.shape[-2]

    performer_probs = softmax_fn(performer_logits).view(-1, V).to(DEVICE_2)
    observer_scores = observer_logits.view(-1, V).to(DEVICE_2)

    xppl = loss_fn(observer_scores, performer_probs).view(-1, S)
    padding_mask = (encoding.input_ids != tokenizer.pad_token_id).type(torch.uint8)

    xppl = (xppl * padding_mask).sum(1) / padding_mask.sum(1)

    return xppl.to("cpu").float().numpy()

def binocular_score(text):
    batch = [text] if isinstance(text, str) else text
    encodings = tokenize(batch)
    observer_logits, performer_logits = get_logits(encodings)
    ppl = perplexity(encodings, observer_logits)
    xppl = cross_perplexity(observer_logits, performer_logits, encodings)

    return (ppl / xppl).tolist()

tokenizer.pad_token = tokenizer.eos_token

In [ ]:
tests = ['''The motivation behind LLM Detection is harm reduction, to trace text origins, block spam, and identify fake news produced by LLMs. *Preemptive detection* methods attempt to "watermark" generated text, but requires full control of the generating models, which already seems to be impossible. Therefore, more recent works have been on *post-hoc detection* methods, which could be used without the cooperation of the text's author. The paper's authors suggest that there are two main groups for post-hoc detectors, the first being finetuning a pretrained language model to perform binary classification. There are many additional techniques that make this approach more effective, but all implementations will require training on text produced by the target model, which is both computationally expensive and limited by the number of new models that are being open-sourced.
The second group uses statistical signatures of machine-generated text, with the aim of zero-shot learning. This would allow for the detection of a wide range of models, with little to no training data. These methods use measures such as perplexity, perplexity curvature, log rank, intrinsic dimensionality, and n-gram analysis. The Binoculars paper proposes a focus on low false positive rate (FPR) and high performance on out-of-domain samples, rather than focusing on classifier AUCs for the high-stakes application of LLM detection.''',
'''In this paper the authors studied reinforcement learning algorithms with nonlinear function approximation. By formulating the problems of value function estimation and policy learning as a bilevel optimization problems, the authors proposed \nQ-learning and actor-critic algorithms that also contains convergence properties, even when nonlinear function approximations are used.  Similar to the stochastic approximation approach adopted by many previous work such as Borkar (https://core.ac.uk/download/pdf/148488247.pdf), they analyze the convergence properties by drawing connections to stability of a two-timescale ODE. Furthermore they also evaluated the effectiveness of the modified Q-learning/actor-critic algorithms on two toy examples.\n\nIn general I find this paper interesting in terms of addressing a long-standing open question of convergence analysis of actor-critic/Q-learning algorithms, when general nonlinear function approximations are used. Through reformulating the problem of value estimation and policy improvement as a bilevel optimization problem, they proposed modifications of Q-learning and actor-critic algorithms, and under certain assumptions they showed that these algorithms converge, which is a non-trivial contribution. \n\nWhile I appreciate the effort of extending existing analysis of these RL algorithms to general nonlinear function approximation,  I find the result of this paper rather incremental. While convergence results are provided, I am not sure how practical are the assumptions listed in the paper. Correct me if i am wrong, it seems that the assumptions are stated for the sake of proving the theoretical results without much practical justifications (especially Assumption 4.3).  Furthermore how can one ensure that these assumptions hold (for example Assumption 4.3 (i) and (ii), especially on the existence of locally stable equilibrium point) ? Unfortunately I haven't had a chance to go over all the proof details, it seems to me the analysis is built upon two-time scale stochastic approximation theory, which is a standard tool in convergence analysis of actor-critic. Since the contribution of this paper is mostly theoretical, can the authors highlight the novel contribution (such as proof techniques used here that are different than that in standard actor-critic analysis from e.g. https://www.semanticscholar.org/paper/Natural-actor-critic-algorithms-Bhatnagar-Sutton/6a40ffc156aea0c9abbd92294d6b729d2e5d5797)  in the main paper?\n\nMy other concern is on the scale of the experiments. While this paper focused on nonlinear function approximation, the examples chosen to evaluate these algorithms are rather small-scale. For example the domains to test Q-learning are standard in RL, and they were previously used to test algorithms with linear function approximation. Can the author compare their results with other existing baselines?''',
'''We the People of the United States, in Order to form a more perfect Union, establish Justice, insure domestic Tranquility, provide for the common defence, promote the general Welfare, and secure the Blessings of Liberty to ourselves and our Posterity, do ordain and establish this Constitution for the United States of America.'''
]
# binocular_score(tests)

In [ ]:
import os
from typing import Union
import numpy as np
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
os.environ['HF_HOME'] = "/data2/cyaiche/hf_cache"
from binoculars import Binoculars

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"


In [ ]:
bino = Binoculars()

In [ ]:
# Vérification à insérer juste avant bino.compute_score(tests)
print(f"Vocabulaire du modèle Performer : {bino.performer_model.config.vocab_size}")

# Simuler ce que fait 'compute_score' pour inspecter les IDs
batch = [tests] if isinstance(tests, str) else tests
encodings = bino._tokenize(batch)
max_token_id = encodings['input_ids'].max().item()

print(f"Token ID maximum dans tes données : {max_token_id}")

if max_token_id >= bino.performer_model.config.vocab_size:
    print("🚨 ERREUR DÉTECTÉE : Tu envoies des Token IDs inconnus au modèle !")

In [ ]:
test = "Ahah! Très bonne remarque! En fait, ce genre d’erreur peut arriver quand une personne écrit un message dans les minutes qui suivent sont réveil. Parfois, il est possible de reconstruire le message de base en se basant sur la phonetique. Mais je suis d’accord avec toi, dans la plupart des cas ça ressemble plus à un ensemble de mots décousu. Je m’en excuse. N’hésites pas à me dire la prochaine fois si je manque de clarté, et je reprendrais tout depuis le début. D’accord?"

In [ ]:
scores = bino.compute_score(test)


In [ ]:
scores

In [ ]:
print(bino.compute_score(tests))
print(np.array(bino.compute_score(tests)))# 0.75661373
print(bino.predict(tests))

In [ ]:
tests2 = ['''This paper proposes an approach for monitoring crop pests and diseases by analyzing community radio content in Uganda using keyword spotting techniques. The authors build models to detect agriculture-related keywords from audio streams in Luganda, a low-resource language.\n\nSummary of key contributions:\n- Collection of agriculture-related keywords from radio shows and online sources\n- Development of keyword spotting models (1D CNN and Siamese CNN) for Luganda\n- Creation of a speech dataset for model training through crowdsourcing\n\nKey reasons for rejection:\n1. Limited technical novelty: The paper primarily applies existing keyword spotting techniques without significant methodological innovations. The approaches used (1D CNN and Siamese CNN) are standard methods that have been previously applied to similar problems.\n\n2. Insufficient experimental validation and analysis:\n- The evaluation is limited to basic accuracy metrics on the keyword detection task\n- No real-world validation of the system's effectiveness for actual crop/pest monitoring\n- Missing ablation studies and comparisons with other approaches\n- Lack of error analysis or discussion of limitations\n- No discussion of computational requirements or deployment considerations\n\nAdditional concerns:\n- The paper lacks clear baselines and comparisons to existing approaches in the field\n- Methodology details are incomplete (e.g., preprocessing parameters, model hyperparameters)\n- Limited discussion of data collection protocols and quality control\n- No analysis of the system's reliability in noisy real-world conditions\n- Missing discussion of ethical considerations regarding radio content monitoring\n\nQuestions for authors:\n1. How does the performance compare to existing crop monitoring systems or other keyword spotting approaches?\n2. What validation was done to ensure the collected keywords are relevant and comprehensive?\n3. How well does the system handle variations in accents, background noise, and speech patterns?\n4. What are the computational requirements for deployment?\n\nWhile the application domain is important and the goal of leveraging community radio for agricultural monitoring is interesting, the current manuscript does not meet the technical standards expected for acceptance. The paper would benefit from more rigorous experimental validation, comparative analysis, and thorough discussion of practical deployment considerations.''',
'''Overall, this paper presents a well-designed and effective approach to video instance segmentation. The use of Inter-frame Communication Transformers (IFC) is a key innovation that enables efficient communication between frames while reducing computational overhead. The authors provide a thorough analysis of the benefits and limitations of their approach, including ablation studies and comparisons to other state-of-the-art methods. The writing is clear and concise, and the paper is well-organized and easy to follow. The experimental results are impressive, demonstrating the effectiveness of the proposed method on challenging datasets. The authors also discuss potential future directions for their work, including integrating temporal information to further improve performance.''']


In [ ]:
scores = bino.compute_score(tests2)


In [ ]:
scores